In [35]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow
import mlflow.xgboost

df = pd.read_csv(r"D:\XGBoost - Project\data\ai4i2020.csv")

rename_map = {
    "Air temperature [K]": "Air_temperature_K",
    "Process temperature [K]": "Process_temperature_K",
    "Rotational speed [rpm]": "Rotational_speed_rpm",
    "Torque [Nm]": "Torque_Nm",
    "Tool wear [min]": "Tool_wear_min"
}
df.rename(columns=rename_map, inplace=True)

drop_cols = ["UDI", "Product ID", "Type", "TWF", "HDF", "PWF", "OSF", "RNF"]
df.drop(columns=drop_cols, inplace=True, errors='ignore')

df["power"] = df["Torque_Nm"] * df["Rotational_speed_rpm"]

X = df.drop(columns=["Machine failure"])
y = df["Machine failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

os.makedirs("models", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
joblib.dump(scaler, "models/scaler.joblib")
pd.DataFrame(X_test_scaled, columns=X.columns).to_csv(r"D:\XGBoost - Project\data\processed\X_test_scaled.csv", index=False)
y_test.to_csv(r"D:\XGBoost - Project\data\processed\y_test.csv", index=False)

params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "learning_rate": 0.1,
    "n_estimators": 100,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

mlflow.set_experiment("predictive_maintenance_xgboost")

with mlflow.start_run() as run:
    mlflow.log_params(params)

    model = xgb.XGBClassifier(**params)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metrics({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_score": f1
    })

    mlflow.xgboost.log_model(model, "xgboost_model")

    model_uri = mlflow.get_artifact_uri("xgboost_model")
    mlflow.register_model(model_uri, "failure_predictor_xgb")

    run_id = run.info.run_id

X_test_scaled_loaded = pd.read_csv(r"D:\XGBoost - Project\data\processed\X_test_scaled.csv").values
y_test_loaded = pd.read_csv(r"D:\XGBoost - Project\data\processed\y_test.csv").values.ravel()

model_loaded = mlflow.xgboost.load_model(f"runs:/{run_id}/xgboost_model")

y_pred_loaded = model_loaded.predict(X_test_scaled_loaded)
acc_loaded = accuracy_score(y_test_loaded, y_pred_loaded)
f1_loaded = f1_score(y_test_loaded, y_pred_loaded)

2026/06/23 22:20:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'failure_predictor_xgb' already exists. Creating a new version of this model...
Created version '14' of model 'failure_predictor_xgb'.
